In [182]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [183]:
import pandas as pd
df = pd.read_csv('train.txt',sep=';',header =  None, names =['text','emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [184]:
df.isnull().sum()

,0
text,0
emotion,0


In [185]:
unique_emotions=df['emotion'].unique()
emotion_numbers={}
i=0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i += 1

df['emotion']=df['emotion'].map(emotion_numbers)

In [186]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [187]:
df['text'] = df['text'].apply(lambda x : x.lower())  # converting every text in lower case

In [188]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))    #remove punctuation

In [189]:
df['text'] = df['text'].apply(remove_punc)

In [190]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)      # remove numbers from the text

In [191]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)       # remove the emojies

In [192]:
# removing stopwords
import nltk

In [193]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [194]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [195]:
stop_words = set(stopwords.words('english'))


In [196]:
df.loc[1]['text']                                            # before removeing stopwords

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [197]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)


In [198]:
df['text'] = df['text'].apply(remove)

In [199]:
df.loc[1]['text']                                             # after removeing stopwords

'go feeling hopeless damned hopeful around someone cares awake'

In [200]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [201]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [202]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [203]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [204]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)


MultinomialNB()

In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.
On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.

In [205]:
y_pred = nb2_model.predict(X_test_tfidf)

In [206]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [207]:
from sklearn.linear_model import LogisticRegression

In [208]:
logistic_model = LogisticRegression(max_iter=1000)

In [209]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.
On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.

In [210]:
log_pred = logistic_model.predict(X_test_tfidf)


In [211]:
print(accuracy_score(y_test,log_pred ))

0.8628125
